# 🐳 Docker — Ultra-Elaborate Mental Models

> **Every section answers four questions: WHY this exists, WHAT it is, HOW it works, WHEN to use it.**
> Real-world scenarios, ❌ before / ✅ after code, and *"Where this is seen in frameworks"* callouts.

---

**Topics**
1. The Container Mental Model — VMs vs Containers
2. Docker Architecture — Daemon, Client, Registry
3. Image Layers — The Union Filesystem
4. Dockerfile Deep Dive — Production Best Practices
5. Multi-Stage Builds — The Artifact Pipeline
6. Docker Compose — Local Development Architecture
7. Networking — Bridge, Host, Overlay
8. Security — Rootless, Read-Only, Capabilities
9. Real-World: How Netflix Containers Work
10. The Docker Architect's Design Framework

---
## 1 · The Container Mental Model — VMs vs Containers

### 🧠 Mental Model — *Shipping Containers for Software*

> **A container is not a lightweight VM. A container is a process (or group of processes) that sees an isolated view of the OS using Linux kernel primitives: namespaces (isolation) and cgroups (resource limits). The isolation is provided by the HOST kernel — not by a hypervisor. This is why containers are fast but share the kernel's attack surface.**

**WHY containers exist:** The "it works on my machine" problem. Before containers:
- Dev runs Python 3.9 on Mac. Prod runs Python 3.7 on CentOS 7.
- Developer installs library X version 1.2. Another service needs version 1.4. Conflict.
- Deployment = SSH into server, manually update packages, hope nothing breaks.
- Rollback = SSH in again, manually downgrade, panic.

**WHAT a container is — Linux primitives:**

```
Linux Kernel Primitives That Make Containers Possible:

NAMESPACES (isolation — what the process can SEE):
  pid namespace  → container has its own PID 1, can't see host processes
  net namespace  → container has its own network stack, IP, ports
  mnt namespace  → container has its own filesystem view
  uts namespace  → container has its own hostname
  ipc namespace  → container has its own IPC (message queues, semaphores)
  user namespace → container can have UID 0 (root) mapped to non-root on host

CGROUPS (resource limits — what the process can USE):
  cpu cgroup   → limits CPU usage (0.5 CPUs, 1 CPU)
  memory cgroup → limits RAM usage (512MB max)
  io cgroup    → limits disk I/O bandwidth
  net cgroup   → limits network bandwidth
```

### VMs vs Containers — The Architecture Difference

```
┌─────── VIRTUAL MACHINES ───────────────────────────────────────┐
│  App A  │  App B  │  App C                                      │
│  Guest OS│  Guest OS│  Guest OS    (full OS per VM, GBs)        │
│  Hypervisor (VMware, KVM, Hyper-V)                              │
│  Host OS                                                         │
│  Hardware                                                        │
└──────────────────────────────────────────────────────────────────┘
Startup: 30-60 seconds | Size: GBs | Isolation: Strong (separate kernel)

┌─────── CONTAINERS ─────────────────────────────────────────────┐
│  App A  │  App B  │  App C                                      │
│  Libs   │  Libs   │  Libs   (only the app + its dependencies)   │
│  Container Runtime (Docker, containerd, CRI-O)                  │
│  Host OS (shared kernel!) ← KEY DIFFERENCE                      │
│  Hardware                                                        │
└──────────────────────────────────────────────────────────────────┘
Startup: milliseconds | Size: MBs | Isolation: Weaker (shared kernel)
```

### The Key Trade-off Table

| Property | Container | VM |
|---|---|---|
| **Startup time** | Milliseconds | 30-60 seconds |
| **Image size** | MBs (10-500MB typical) | GBs (5-20GB) |
| **Process density** | 100s per host | 10s per host |
| **Kernel isolation** | Shared (weaker security) | Separate (stronger security) |
| **OS portability** | Linux containers on Linux | Any OS on any OS (via hypervisor) |
| **Live migration** | Easy (just move the image) | Harder |
| **Persistent storage** | Volumes (explicit) | Disk (implicit) |

### 🌍 Real-World: Why Containers Changed Cloud Economics
Before containers: a "4-core, 8GB" VM ran one app. Utilization ≈ 10-15% because the app needed 4 cores at peak but sat idle most of the day.

With containers on Kubernetes: that same VM now runs 20+ containers, bin-packed by resource requests. Utilization → 60-80%. **Same hardware, 4-5× more workloads.** This is why AWS/GCP/Azure pushed hard for container adoption — their utilization numbers improved dramatically when customers containerized.

### ⚠️ The Security Implication
Container isolation is weaker than VM isolation because **they share the host kernel**. A kernel vulnerability (like Dirty COW, 2016) can break container isolation. This is why:
- **Never run containers as root** (if container process escapes to host, it's root on the host)
- **Use read-only root filesystems** where possible
- **Drop capabilities** (CAP_NET_ADMIN, CAP_SYS_PTRACE are dangerous)
- **Use gVisor or Kata Containers** for truly multi-tenant sensitive workloads (they add a kernel boundary)

---
## 2 · Docker Architecture

### 🧠 Mental Model — *The Client-Daemon-Registry Triangle*

> **Docker is a client-server architecture. The CLI (client) sends commands to the daemon (server). The daemon builds images, runs containers, and communicates with registries. The image is the artifact; the container is the running instance; the registry is the artifact store. One image → many containers. Like a class → many instances.**

### Architecture Overview

```
Developer Machine                    Registry
┌───────────────────┐              ┌───────────────────┐
│  Docker Client    │──push/pull──▶│ Docker Hub        │
│  (docker CLI)     │              │ GitHub GHCR       │
│        │          │              │ AWS ECR / GCR     │
│        │ REST API │              └───────────────────┘
│        ▼          │
│  Docker Daemon    │
│  (dockerd)        │◀── containerd ◀── runc (OCI runtime)
│  - Build images   │                   (actually creates the container)
│  - Run containers │
│  - Manage volumes │
│  - Manage networks│
└───────────────────┘
```

### The Runtime Stack (from docker run to process)

```
docker run nginx
     ↓
Docker CLI (sends API request)
     ↓
dockerd (Docker daemon — manages the request)
     ↓
containerd (manages container lifecycle)
     ↓
containerd-shim (stays alive to hold the container's stdio)
     ↓
runc (OCI runtime — sets up namespaces/cgroups, exec's the process)
     ↓
nginx process (PID 1 in the container's PID namespace)
```

**Why the layered runtime matters:** Kubernetes bypasses Docker entirely — it talks directly to **containerd** (or CRI-O). The Dockershim was removed in K8s 1.24. `docker` CLI is a developer convenience; the production runtime is containerd.

### Image vs Container

```
Image  = Blueprint (read-only, immutable, layered filesystem)
         Like a class definition, a VM snapshot, an AMI

Container = Running instance (image + thin read-write layer on top)
           Like an object instance, a running VM

docker pull nginx:1.25   # downloads image layers from registry
docker run nginx:1.25    # creates container from image
                         # → adds writable layer on top of image layers
                         # → all writes go to this layer (ephemeral!)
docker stop container1   # container removed, writable layer GONE
                         # image still intact for next docker run
```

---
## 3 · Image Layers — The Union Filesystem

### 🧠 Mental Model — *A Stack of Transparent Slides*

> **A Docker image is a stack of read-only layers. When you read a file, Docker looks through the layers from top to bottom until it finds it. When you write a file, the write goes to a new layer on top (copy-on-write). The genius: identical layers are shared across all images and all containers that use them. 10 containers running nginx share one copy of the nginx layer in memory and on disk.**

### Layer Visualization

```
Image: myapp:latest
┌──────────────────────────────────────────────────────┐
│ Layer 5 (COPY ./app /app)          8 MB  ← YOUR CODE │
│ Layer 4 (RUN pip install -r req)  150 MB ← DEPS      │
│ Layer 3 (COPY requirements.txt)     1 KB ← LOCKFILE  │
│ Layer 2 (RUN apt-get install curl)  5 MB ← OS TOOLS  │
│ Layer 1 (FROM python:3.11-slim)   120 MB ← BASE IMAGE│
└──────────────────────────────────────────────────────┘
Total: ~284 MB

Container: myapp (running)
┌──────────────────────────────────────────────────────┐
│ WRITABLE LAYER (container writes go here)  ephemeral │
│─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─│
│ Layer 5 (read-only)                                  │
│ Layer 4 (read-only) ← SHARED with all python:3.11   │
│ Layer 3 (read-only)   containers if same req.txt     │
│ Layer 2 (read-only)                                  │
│ Layer 1 (read-only)                                  │
└──────────────────────────────────────────────────────┘
```

### Why Layer Order in Dockerfile Matters for Build Speed

```
Cache invalidation rule: if a layer changes, ALL layers below it are rebuilt.

❌ SLOW: Code change → pip install runs every time (150 seconds!)
  1. FROM python:3.11-slim
  2. COPY . /app              ← entire app copied first
  3. RUN pip install -r /app/requirements.txt  ← invalidated by any code change

✅ FAST: Code change → only code layer rebuilt (2 seconds!)
  1. FROM python:3.11-slim
  2. COPY requirements.txt /app/  ← only lockfile, rarely changes
  3. RUN pip install -r /app/requirements.txt  ← cached unless req.txt changes
  4. COPY . /app               ← code changes here, only this layer rebuilds

The rule: MOST STABLE → TOP. MOST FREQUENTLY CHANGING → BOTTOM.
```

In [ ]:
"""
Docker Layer Cache Simulator
============================
Demonstrates why Dockerfile instruction ORDER dramatically affects
build times in a CI pipeline.

This model is why 'cache the pip install' is the #1 Docker optimization
for Python services.
"""
from __future__ import annotations
import hashlib
import time
from dataclasses import dataclass, field
from typing import Optional, List


@dataclass
class Layer:
    instruction: str
    content_hash: str      # hash of inputs (files, commands)
    build_time_s: float    # how long this layer takes to build
    size_mb: float


class DockerBuildSimulator:
    """Simulates Docker's layer cache mechanism."""

    def __init__(self):
        self._cache: dict[str, Layer] = {}  # content_hash → cached layer

    def build(self, layers: List[Layer], description: str) -> dict:
        """Build an image, using cache where possible."""
        total_time = 0.0
        cache_hits = 0
        cache_miss_started = False  # once cache breaks, all subsequent rebuild
        results = []

        for layer in layers:
            if not cache_miss_started and layer.content_hash in self._cache:
                status = "CACHED"
                time_taken = 0.001  # reading from cache is near-instant
                cache_hits += 1
            else:
                status = "BUILDING"
                time_taken = layer.build_time_s
                cache_miss_started = True  # all layers after miss must rebuild
                self._cache[layer.content_hash] = layer

            total_time += time_taken
            results.append({
                "instruction": layer.instruction,
                "status": status,
                "time": time_taken,
                "size": layer.size_mb,
            })

        return {
            "description": description,
            "total_time_s": total_time,
            "cache_hits": cache_hits,
            "layers": results,
        }

    def print_build(self, result: dict) -> None:
        print(f"\n=== {result['description']} ===")
        for r in result["layers"]:
            status_icon = "📦" if r["status"] == "CACHED" else "🔨"
            print(f"  {status_icon} {r['status']:7s} ({r['time']:.1f}s) {r['instruction']:40s} [{r['size']:.0f}MB]")
        print(f"  {'─'*70}")
        print(f"  Total build time: {result['total_time_s']:.1f}s | Cache hits: {result['cache_hits']}")

In [ ]:
# Simulate two builds: one with bad layer order, one with optimized order
# Scenario: engineer changes ONE Python file between Build 1 and Build 2

simulator = DockerBuildSimulator()

# ──── Build 1: First build (nothing cached) ────────────────────────────

# ❌ BAD ORDER: copy code before installing dependencies
bad_layers_build1 = [
    Layer("FROM python:3.11-slim",              hash("python:3.11-slim"),     0.5,  127.0),
    Layer("COPY . /app",                         hash("app_v1_content"),       0.2,    5.0),  # all app files
    Layer("RUN pip install -r requirements.txt", hash("requirements_v1+app_v1"), 120.0, 150.0),  # invalidated by any code change!
    Layer("CMD python app.py",                   hash("cmd"),                  0.1,    0.0),
]

# ✅ GOOD ORDER: dependencies before code
good_layers_build1 = [
    Layer("FROM python:3.11-slim",              hash("python:3.11-slim"),     0.5,  127.0),
    Layer("COPY requirements.txt /app/",         hash("req_v1"),               0.1,    0.1),  # only lockfile
    Layer("RUN pip install -r requirements.txt", hash("req_v1"),               120.0, 150.0),  # keyed only on lockfile
    Layer("COPY . /app",                         hash("app_v1_content"),       0.2,    5.0),  # code last
    Layer("CMD python app.py",                   hash("cmd"),                  0.1,    0.0),
]

r1_bad  = simulator.build(bad_layers_build1,  "BAD ORDER — Build 1 (first build, no cache)")
r1_good = simulator.build(good_layers_build1, "GOOD ORDER — Build 1 (first build, no cache)")
simulator.print_build(r1_bad)
simulator.print_build(r1_good)

print("\n" + "═"*72)
print("  Engineer changes one line of Python code...")
print("═"*72)

# ──── Build 2: After changing ONE .py file ────────────────────────────
bad_layers_build2 = [
    Layer("FROM python:3.11-slim",              hash("python:3.11-slim"),     0.5,  127.0),  # cached
    Layer("COPY . /app",                         hash("app_v2_content"),       0.2,    5.0),  # CHANGED → cache miss
    Layer("RUN pip install -r requirements.txt", hash("requirements_v1+app_v2"), 120.0, 150.0),  # REBUILT despite no dep change!
    Layer("CMD python app.py",                   hash("cmd"),                  0.1,    0.0),
]

good_layers_build2 = [
    Layer("FROM python:3.11-slim",              hash("python:3.11-slim"),     0.5,  127.0),  # cached
    Layer("COPY requirements.txt /app/",         hash("req_v1"),               0.1,    0.1),  # cached (req didn't change)
    Layer("RUN pip install -r requirements.txt", hash("req_v1"),               120.0, 150.0),  # CACHED!
    Layer("COPY . /app",                         hash("app_v2_content"),       0.2,    5.0),  # only this rebuilds
    Layer("CMD python app.py",                   hash("cmd"),                  0.1,    0.0),
]

r2_bad  = simulator.build(bad_layers_build2,  "BAD ORDER — Build 2 (after code change)")
r2_good = simulator.build(good_layers_build2, "GOOD ORDER — Build 2 (after code change)")
simulator.print_build(r2_bad)
simulator.print_build(r2_good)

speedup = r2_bad['total_time_s'] / r2_good['total_time_s']
print(f"\n⚡ Layer ordering speedup on incremental builds: {speedup:.0f}×")
print(f"   Bad: {r2_bad['total_time_s']:.0f}s | Good: {r2_good['total_time_s']:.1f}s")
print(f"   In a team of 10 engineers, each pushing 5x/day: this saves")
print(f"   {(r2_bad['total_time_s'] - r2_good['total_time_s']) * 50 / 3600:.1f} engineer-hours per day of waiting.")

---
## 4 · Dockerfile Deep Dive — Production Best Practices

### 🧠 Mental Model — *The Dockerfile is Code — Review It Like Code*

> **A Dockerfile is the source of truth for your deployment artifact. Every anti-pattern in a Dockerfile ships to production and runs in your Kubernetes cluster. Security vulnerabilities in your base image are YOUR CVEs. Large image sizes slow your cluster autoscaler. A 5-minute Dockerfile review prevents days of production debugging.**

### Complete Production Dockerfile — Python Service

```dockerfile
# ✅ PRODUCTION DOCKERFILE
# Every decision here is explained below

# ── Stage 1: Builder ──────────────────────────────────────────────────
# Use specific version tag — never 'latest' (non-deterministic)
FROM python:3.11.9-slim-bookworm AS builder

# Set working directory (always explicit — don't rely on default)
WORKDIR /build

# Install build tools needed for compilation (kept in builder, not final image)
RUN apt-get update && apt-get install -y --no-install-recommends \
    gcc \
    libpq-dev \
    && rm -rf /var/lib/apt/lists/*  # ← clean apt cache in same layer!

# Copy lockfile FIRST (most stable → cached most often)
COPY requirements.txt .

# Install to a specific prefix so we can copy only what's needed
RUN pip install --prefix=/install --no-cache-dir -r requirements.txt

# ── Stage 2: Runtime (final image) ───────────────────────────────────
FROM python:3.11.9-slim-bookworm AS runtime

# ✅ Create non-root user — CRITICAL for security
RUN groupadd --gid 1001 appuser && \
    useradd --uid 1001 --gid appuser --shell /bin/bash --create-home appuser

WORKDIR /app

# Copy only compiled packages from builder (no gcc, no build tools)
COPY --from=builder /install /usr/local

# Copy application code
COPY --chown=appuser:appuser src/ ./src/
COPY --chown=appuser:appuser main.py .

# ✅ Switch to non-root before CMD
USER appuser

# ✅ Health check — Kubernetes uses this for readiness
HEALTHCHECK --interval=30s --timeout=5s --start-period=5s --retries=3 \
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"

# ✅ Expose port (documentation only — doesn't actually publish)
EXPOSE 8000

# ✅ Use ENTRYPOINT + CMD pattern:
#   ENTRYPOINT = the command (can't be overridden accidentally)
#   CMD = default arguments (can be overridden with docker run args)
ENTRYPOINT ["python", "-m", "uvicorn"]
CMD ["main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### Anti-Pattern Audit

| Anti-Pattern | Risk | Fix |
|---|---|---|
| `FROM ubuntu:latest` | Non-deterministic, grows over time | Pin: `FROM ubuntu:22.04` |
| `RUN apt-get update` on its own layer | Stale cache: update cached, install runs against old metadata | Combine: `RUN apt-get update && apt-get install -y ...` |
| `RUN pip install package` without version | Different version on every build | `requirements.txt` with pinned versions |
| No `.dockerignore` | `.git/`, `node_modules/`, secrets copied into image | Create `.dockerignore` matching `.gitignore` |
| `USER root` or no USER directive | Container runs as root — kernel escape = root on host | `USER nonroot` or create app user |
| `COPY . .` before `pip install` | Every code change invalidates dependency cache | Copy lockfile first, then code |
| `ENV SECRET_KEY=hardcoded` | Secret baked into image, visible in `docker inspect` | Use runtime env vars or secrets manager |
| No multi-stage build | Build tools (gcc, maven) shipped in production image | Multi-stage: build in Stage 1, copy artifacts to Stage 2 |

In [ ]:
"""
Dockerfile Auditor
==================
Static analysis tool for Dockerfiles — the DevOps equivalent of a linter.
This is the kind of tool a platform team builds to enforce standards
across all microservice Dockerfiles in an organization.
"""
from __future__ import annotations
import re
from dataclasses import dataclass
from enum import Enum
from typing import List


class Severity(Enum):
    CRITICAL = "CRITICAL"
    HIGH     = "HIGH"
    MEDIUM   = "MEDIUM"
    INFO     = "INFO"


@dataclass
class DockerFinding:
    rule: str
    severity: Severity
    line: int
    message: str
    fix: str


class DockerfileAuditor:
    RULES = [
        {
            "name": "LATEST_TAG",
            "pattern": r"^FROM\s+[^:]+:latest",
            "severity": Severity.HIGH,
            "message": "'latest' tag is non-deterministic — build will differ over time",
            "fix": "Pin to specific version: FROM python:3.11.9-slim-bookworm",
        },
        {
            "name": "NO_VERSION_TAG",
            "pattern": r"^FROM\s+\S+$",  # FROM image with no tag at all
            "severity": Severity.HIGH,
            "message": "No version tag specified — defaults to 'latest'",
            "fix": "Always specify a version tag: FROM nginx:1.25.3",
        },
        {
            "name": "ROOT_USER",
            "pattern": r"USER\s+root",
            "severity": Severity.CRITICAL,
            "message": "Explicitly switching to root user",
            "fix": "Create a non-root user: RUN useradd -m appuser && USER appuser",
        },
        {
            "name": "NO_USER_DIRECTIVE",
            "pattern": None,  # special check
            "severity": Severity.CRITICAL,
            "message": "No USER directive — container runs as root by default",
            "fix": "Add 'USER nonroot' or create and use a specific app user",
        },
        {
            "name": "APT_UPDATE_SEPARATE",
            "pattern": r"^RUN\s+apt-get\s+update\s*$",
            "severity": Severity.MEDIUM,
            "message": "apt-get update on its own layer causes stale cache issues",
            "fix": "Combine: RUN apt-get update && apt-get install -y ... && rm -rf /var/lib/apt/lists/*",
        },
        {
            "name": "SECRET_IN_ENV",
            "pattern": r"ENV\s+(SECRET|PASSWORD|TOKEN|KEY|PASS)\s*=\s*\S+",
            "severity": Severity.CRITICAL,
            "message": "Secret hardcoded in ENV — baked into image, visible in 'docker inspect'",
            "fix": "Pass secrets at runtime: -e SECRET=$SECRET or use secrets manager",
        },
        {
            "name": "ADD_INSTEAD_OF_COPY",
            "pattern": r"^ADD\s+(?!http)",  # ADD with local files (COPY is safer)
            "severity": Severity.MEDIUM,
            "message": "ADD with local files — use COPY instead (ADD has hidden behaviors: auto-extract tarballs)",
            "fix": "Use COPY for local files. Reserve ADD only for URLs or tar auto-extraction.",
        },
        {
            "name": "PIP_NO_CACHE",
            "pattern": r"pip install(?!.*--no-cache-dir)",
            "severity": Severity.INFO,
            "message": "pip install without --no-cache-dir grows image unnecessarily",
            "fix": "Add --no-cache-dir to pip install",
        },
    ]

    def audit(self, dockerfile: str) -> List[DockerFinding]:
        findings = []
        lines = dockerfile.strip().split("\n")
        has_user_directive = any(l.strip().startswith("USER ") for l in lines)

        for lineno, line in enumerate(lines, 1):
            line = line.strip()
            if line.startswith("#") or not line:
                continue
            for rule in self.RULES:
                if rule["pattern"] is None:  # special case
                    continue
                if re.search(rule["pattern"], line, re.IGNORECASE):
                    findings.append(DockerFinding(
                        rule=rule["name"], severity=rule["severity"],
                        line=lineno, message=rule["message"], fix=rule["fix"],
                    ))

        # Check for missing USER directive
        if not has_user_directive:
            no_user_rule = next(r for r in self.RULES if r["name"] == "NO_USER_DIRECTIVE")
            findings.append(DockerFinding(
                rule="NO_USER_DIRECTIVE", severity=Severity.CRITICAL,
                line=0, message=no_user_rule["message"], fix=no_user_rule["fix"],
            ))

        return sorted(findings, key=lambda f: ["CRITICAL","HIGH","MEDIUM","INFO"].index(f.severity.value))


BAD_DOCKERFILE = """
FROM ubuntu:latest
RUN apt-get update
RUN apt-get install -y python3 pip curl
ADD . /app
ENV SECRET_KEY=supersecretvalue123
RUN pip install fastapi uvicorn
CMD ["python3", "-m", "uvicorn", "main:app"]
"""

auditor  = DockerfileAuditor()
findings = auditor.audit(BAD_DOCKERFILE)

print("=== Dockerfile Security Audit ===")
print(f"Found {len(findings)} issue(s)\n")
for f in findings:
    line_str = f"line {f.line}" if f.line else "(global)"
    print(f"[{f.severity.value:8s}] {f.rule} ({line_str})")
    print(f"  Issue: {f.message}")
    print(f"  Fix:   {f.fix}")
    print()

---
## 5 · Multi-Stage Builds — The Artifact Pipeline

### 🧠 Mental Model — *Compile in a Big Factory, Ship Only the Product*

> **Multi-stage builds let you use one Docker image (with all build tools) to compile your application, then copy ONLY the compiled artifact into a minimal final image. You don't ship the factory floor to customers — you ship the product. This reduces attack surface and image size simultaneously.**

### Before/After: Go Service

```
❌ BEFORE: Single-stage build
  FROM golang:1.21           # 800MB image
  COPY . .
  RUN go build -o /app .
  CMD ["/app"]
  → Final image: 800MB (Go compiler + source code shipped to prod)

✅ AFTER: Multi-stage build
  FROM golang:1.21 AS builder    # 800MB, but only used in CI
  COPY . .
  RUN CGO_ENABLED=0 go build -o /app .

  FROM scratch                   # Empty image — literally nothing
  COPY --from=builder /app /app
  CMD ["/app"]
  → Final image: ~8MB (just the binary)

  Reduction: 800MB → 8MB = 100× smaller
  Security: no shell, no package manager, no compiler in prod image
```

### Size Reduction by Language

| Language | Without Multi-Stage | With Multi-Stage | Reduction |
|---|---|---|---|
| **Go** | 800MB | 8-30MB | 30-100× |
| **Java** | 800MB (JDK + sources) | 100-200MB (JRE + JAR) | 4-8× |
| **Node.js** | 1.2GB (with devDeps) | 150-300MB (prod deps only) | 4-8× |
| **Python** | 800MB (with build tools) | 150-250MB (slim + app) | 3-5× |
| **Rust** | 2GB (Rust toolchain) | 5-50MB (static binary) | 40-400× |

### Why Image Size Matters in Production

```
Smaller images =
  1. Faster pull from registry → faster pod startup → faster autoscaling response
  2. Less network bandwidth → lower ECR/GCR data transfer costs
  3. Less disk on node → more pods per node → lower cluster cost
  4. Smaller attack surface → fewer CVEs → less security debt

Real numbers (team of 50 engineers, 20 pushes/day, 5 services):
  800MB image × 100 pulls/day = 80GB/day of registry traffic
  50MB image  × 100 pulls/day = 5GB/day of registry traffic
  Savings: 75GB/day × $0.09/GB (ECR data transfer) ≈ $200/day = $73,000/year
```

---

## 6 · Security — Rootless, Read-Only, Capabilities

### The Security Defense-in-Depth Model for Containers

```
Attack scenario: Attacker exploits a vulnerability in your Python app.
They have code execution inside the container.

Without security hardening:
  Container process = root (UID 0)
  Attacker can: write to filesystem, install tools, access other containers,
                exploit kernel vulnerability to escape to host as root

With defense in depth:
  Layer 1: Non-root user (UID 1001)
           → Even if attacker has code exec, they're a low-priv user
  Layer 2: Read-only root filesystem (--read-only)
           → Can't write to disk, can't install tools or modify code
  Layer 3: Dropped capabilities (--cap-drop ALL)
           → Can't bind port <1024, can't modify kernel parameters
  Layer 4: No new privileges (--security-opt no-new-privileges)
           → Can't escalate via setuid binaries
  Layer 5: Seccomp profile
           → Limits which syscalls the process can make
  Layer 6: Network policy (Kubernetes)
           → Can only talk to services it's allowed to reach

Attacker's code execution is now: constrained to a tiny, isolated space
with no write access, no privileges, and limited network reach.
```

### 🌍 Where This Is Seen in Production
- **Google Kubernetes Engine:** All GKE Autopilot nodes enforce no-root containers by default
- **OpenShift (Red Hat):** Runs ALL containers as non-root by default — images that require root fail
- **AWS Fargate:** Strict seccomp profiles applied to all tasks
- **GitHub Actions runners:** Jobs run as non-root with capabilities dropped

---
## 7 · The Docker Architect's Design Framework

### Before Writing a Dockerfile for a New Service

```
1. BASE IMAGE
   - Use official slim/alpine variants (smaller attack surface)
   - Pin to specific version SHA, not tag
   - Subscribe to Dependabot/Renovate for base image updates
   
2. LAYER STRATEGY
   - Dependencies BEFORE code (cache optimization)
   - Minimize layers where beneficial
   - Clean package caches in the same RUN command

3. MULTI-STAGE
   - Always separate build dependencies from runtime
   - Name your stages (AS builder, AS runtime)
   - Copy only what the runtime needs

4. SECURITY
   - Non-root user (create in Dockerfile)
   - Minimal capabilities
   - No secrets in image (ENV, COPY of .env files)
   - Read-only filesystem where possible

5. OBSERVABILITY
   - HEALTHCHECK instruction (K8s uses for readiness)
   - App logs to stdout/stderr (never to files in container)
   - Structured JSON logging

6. BUILD PERFORMANCE  
   - .dockerignore file (exclude .git, node_modules, __pycache__)
   - BuildKit enabled (DOCKER_BUILDKIT=1)
   - Registry cache for CI (--cache-from, --cache-to)

7. IMAGE SCANNING
   - Scan image in CI pipeline (trivy, grype, Snyk)
   - Block on CRITICAL CVEs
   - Re-scan images weekly even without code changes
```

### 📚 What to Study Next

1. **Module 04 — Kubernetes:** Where containers live in production. Understanding K8s requires understanding Docker images and the container runtime.
2. **`examples/` folder:** Docker Compose for local dev, networking deep dive, volume patterns
3. **examples/03_docker_compose.ipynb:** The local development environment architecture